# Station charges — 01 source extraction

Single source of truth for the station charge source register: what an
infrastructure manager or station operator charges for a passenger train
calling at a station. Every row is written by this notebook;
`data/sources_register.csv` is a generated artifact and must never be
hand-edited.

Run this before `02_station_charges.ipynb`, which fails rather than citing a
source that does not resolve against this register.

Same contract as `tac/calib/`, `energy_pricing/calib/` and `facility/calib/`:
notebook is truth, CSV is output. Put the documents themselves in `sources/`
(gitignored — they are publishers' PDFs and spreadsheets, not ours to
redistribute) and register them here by filename.

In [1]:
# Station charges — source extraction
#
# Station charge tariffs come from three kinds of document, and each is read
# differently in 02: a station price list (usually PDF), a network statement
# annex (PDF or XLSX), and figures transcribed by hand from a source that
# cannot be parsed at all. The register does not care which — it records what
# the document is, so a value can always be traced back to it.

import csv
from pathlib import Path


def _resolve_data_dir() -> Path:
    """Notebook may run from charges/ or from the repo root; resolve either."""
    here = Path.cwd()
    for cand in (
        here / "data",
        here / "backend/models/infrastructure/stops/charges/data",
    ):
        if cand.parent.exists():
            cand.mkdir(exist_ok=True)
            return cand
    raise RuntimeError(f"cannot locate the charges data directory from {here}")


DATA_DIR = _resolve_data_dir()
SOURCES_DIR = DATA_DIR.parent / "sources"
SOURCES_DIR.mkdir(exist_ok=True)
print(f"data directory:    {DATA_DIR}")
print(f"source documents:  {SOURCES_DIR}")

# Date the register was last reviewed end to end.
REGISTER_REVIEWED = "TO_VERIFY"

REGISTER_COLUMNS = [
    "source_id",
    "short_id",
    "used",
    "downloaded",
    "title",
    "publisher",
    "pub_year",
    "price_basis_year",
    "currency",
    "kind",
    "url_or_file",
    "date_accessed",
    "reliability_note",
]

register_rows: list[tuple] = []

data directory:    C:\Users\david\PycharmProjects\night-train-target-network\backend\models\infrastructure\stops\charges\data
source documents:  C:\Users\david\PycharmProjects\night-train-target-network\backend\models\infrastructure\stops\charges\sources


## Station price lists and network statement annexes

One row per document. `kind` drives nothing mechanically but tells the reader
in `02` which extraction path a document needs:

| kind | typical format | how `02` reads it |
|---|---|---|
| `station_price_list` | PDF | `pdf_table` if the tables are machine-readable, otherwise `manual` |
| `network_statement` | PDF or XLSX | `xlsx_table` for annexes published as spreadsheets |
| `operator_model` | XLSX | `xlsx_table` |
| `manual_transcription` | anything | `manual` — figures typed into the notebook, with the page cited |

`used` is `Used` once a value in `02` cites it, `Registered` while it is only
listed. `downloaded` is `x` when the file is in `sources/`.

**Josh:** add your sources here first, then read them in `02`. Leave
`price_basis_year` as the year the tariff applies to, not the year of
publication — the two differ in most network statements.

In [2]:
# --- Station charge sources ---
_R = REGISTER_REVIEWED

register_rows += [
    (
        "DE-DB-SPL-2026",
        "de_db_spl_2026",
        "Used",
        "",
        "Stationspreisliste 2026",
        "DB InfraGO AG",
        2026,
        2026,
        "EUR",
        "station_price_list",
        "de_db_stationspreisliste_2026.pdf",
        "2026-08-25",
        (
            "Per-call station charge for all 5,412 German stations, published as "
            "an SPNV (regional) and an SPFV (long-distance) share per station. "
            "The catalog takes the SPFV share: a night train is long-distance. "
            "Extracted from the PDF and matched to the catalog's 105 German "
            "stops, all of which the list covers. Stand 18.06.2026, valid from "
            "01.01.2026. Figures are net of VAT (confirmed 2026-08-25), matching "
            "the project convention."
        ),
    ),
    # --- 2026-09-10 batch (Josua), FR re-sourced 2026-09-13 (David). One row per
    #     document; the reliability_note says what was taken and what was assumed.
    (
        "AT-ÖBB-SNNB-2026",
        "at_oebb_snnb_2026",
        "Used",
        "",
        (
            "Schienennetz-Nutzungsbedingungen 2026 — Entgelt Verkehrsstation, mit "
            "Verzeichnis der Verkehrsstationen 2026 (2.3.3)"
        ),
        "ÖBB-Infrastruktur AG",
        2026,
        2026,
        "EUR",
        "network_statement",
        (
            " "
            "https://infrastruktur.oebb.at/de/geschaeftspartner/schienennetz/snnb/snnb-2026/schienennetz-nutzungsbedingungen-2026.pdf"
        ),
        "2026-09-10",
        (
            "Basisleistung Verkehrsstation, Produkt 3.a, one price per stop by "
            "station category A-D; the category of each stop from the 2.3.3 station "
            "register. Optional services (sales space, ticket machines) excluded. "
            "Transcribed by Josua."
        ),
    ),
    (
        "BE-SNCB-SPSA-2026-A2",
        "be_sncb_spsa_2026",
        "Used",
        "",
        "SPSA 2026, Annexe 2 — tarifs des services en gare",
        "SNCB/NMBS",
        2026,
        2026,
        "EUR",
        "network_statement",
        (
            " "
            "https://www.belgiantrain.be/-/media/corporate/services/rrs/docs2026/spsa-fr-a2-2026.ashx"
        ),
        "2026-09-10",
        (
            "Passenger-information service per effective stop, station categories "
            "S/M/L. Station access itself is a flat 1 EUR per access request and "
            "timetable year, not allocable per stop. 21% is the Belgian standard "
            "rate; the SPSA names no single rate."
        ),
    ),
    (
        "BG-NRIC-NS-2025-2026-ANNEX-5.3.2-v06",
        "bg_nric_ns_2026",
        "Used",
        "",
        "Network Statement 2025/2026, Annex 5.3.2 v06 (18.03.2026)",
        "NRIC (НКЖИ)",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://ten-t.rail-infra.bg/bg/392",
        "2026-09-10",
        (
            "Service 2.4, use of passenger stations, is not charged; only "
            "commercial floor space (service 2.3) is priced per m2 and month. Every "
            "Bulgarian stop is recorded as not levied."
        ),
    ),
    (
        "CH-BAV-NZV-BAV-2026-ANNEX2",
        "ch_bav_nzv_2026",
        "Used",
        "",
        "NZV-BAV (SR 742.122.4), Anhang 2, Stand 01.02.2026",
        "Bundesamt für Verkehr",
        2026,
        2026,
        "CHF",
        "network_statement",
        "https://ch.odat.ch/de/cc/742.122.4-20260201-de.html",
        "2026-09-10",
        (
            "Demand-related Haltezuschlag of CHF 2.00 per ordered stop on the "
            "mixed-traffic lines and stations listed in Anhang 2; platform access "
            "is part of the Swiss track-access base price. Stops absent from Anhang "
            "2 are not levied. Converted at ECB 09.09.2026, EUR 1 = CHF 0.9404."
        ),
    ),
    (
        "CZ-SZ-NS-2026-ANNEX-C-II.5",
        "cz_sz_ns_2026",
        "Registered",
        "",
        "Prohlášení o dráze 2026, Annex C II.5 — access paths for passengers",
        "Správa železnic",
        2026,
        2026,
        "CZK",
        "network_statement",
        "https://www.spravazeleznic.cz/dopravci/prohlaseni-o-draze",
        "2026-09-10",
        (
            "Per stop AND per tonne: 0.04-0.08 CZK per scheduled stop per tonne of "
            "train mass excluding non-carrying traction (mpk), by station category "
            "11-15. Not a static per-stop figure. sources/cz_station_charges.csv "
            "carries the per-tonne unit rate as basis per_stop_per_tonne and is "
            "deliberately NOT in 02's CHARGE_FILES until the cost model prices it "
            "from CompositionType.total_weight_t(). Converted at ECB 09.09.2026, "
            "EUR 1 = CZK 24.247."
        ),
    ),
    (
        "DK-BANEDANMARK-NETWORK-STATEMENT-2026",
        "dk_bdk_ns_2026",
        "Used",
        "",
        "Network Statement 2026",
        "Banedanmark",
        2026,
        2026,
        "DKK",
        "network_statement",
        "https://www.bane.dk/en/Rail-companies/Network-Statement",
        "2026-09-10",
        (
            "No separately published per-stop passenger-station charge; "
            "infrastructure charges are per train-km plus the Storebaelt and "
            "Oresund bridge charges. Every Danish stop is recorded as not levied."
        ),
    ),
    (
        "ES-ADIF-CANON-ESTACIONES-2026",
        "es_adif_canon_2026",
        "Used",
        "",
        (
            "Canon por utilización de estaciones 2026 (Resolución BOE 04.11.2025) "
            "with Relación de instalaciones de servicio 2026"
        ),
        "ADIF / ADIF Alta Velocidad",
        2026,
        2026,
        "EUR",
        "network_statement",
        "https://www.boe.es/eli/es/res/2025/11/04/(2)/dof/spa/pdf",
        "2026-09-10",
        (
            "Fixed stop component for Larga Distancia at an intermediate stop, by "
            "station category 1-5: 44.74 / 41.64 / 37.32 / 20.57 / 8.88 EUR. Origin "
            "and destination stops are priced higher; the passenger-based intensity "
            "component (0.40 EUR per boarding or alighting passenger) is not "
            "included because it needs monthly station-specific ridership."
        ),
    ),
    (
        "FR-GC-DRG-2024-A1",
        "fr_gc_drg_2024",
        "Used",
        "",
        (
            "Document de référence des gares 2024, version saisine — Annexe A1 "
            "Barème tarifaire (Prestation de base unifiée) and Annexe A0.1 Liste "
            "des gares"
        ),
        "SNCF Gares & Connexions",
        2024,
        2024,
        "EUR",
        "network_statement",
        (
            "Annexe_A0_1-Liste_des_gares_DRG_2024_v_saisine.xlsx; "
            "Annexe_A1_1-Bareme_tarifaire_DRG_2024_v_saisine.xlsx "
            "(https://www.garesetconnexions.sncf/en/rail-companies/stations-statement)"
        ),
        "2026-09-13",
        (
            "Per departing train, EUR HT, from the 'Autres trains' column; the "
            "TER/Transilien column is the conventioned regional rate. The tariff "
            "perimeter is station category A/B/C by region, or a station-specific "
            "TGA perimeter for the 20 largest stations. Catalog stops matched on "
            "UIC code (7-digit OSM uic_ref against the 8-digit UIC complet): 96 of "
            "96, no name/UIC disagreement. TGA SUD PARIS priced at the Austerlitz / "
            "Bercy / Gare de Lyon / Montparnasse Hall 1 rate. 2024 is the latest "
            "annex on hand; the 2026 DRG should replace it. 20% standard VAT "
            "applied. Supersedes the 2026-09-10 assumption that France levies no "
            "station charge."
        ),
    ),
    (
        "HU-VPE-HUSZ-2025-2026",
        "hu_vpe_husz_2026",
        "Used",
        "",
        "Hálózati Üzletszabályzat (HÜSZ) 2025/2026",
        "VPE Vasúti Pályakapacitás-elosztó Kft.",
        2025,
        2026,
        "HUF",
        "network_statement",
        "https://vpe.kti.hu/halozati-uzletszabalyzat-husz",
        "2026-09-10",
        (
            "Consolidated station-stop amount of 3974 HUF per stop (the 2024/25 "
            "amount times the 1.0245 non-energy uplift). HÜSZ 2025/26 separates "
            "passenger-information services from the former station-use bundle; the "
            "consolidated figure is used so the file stays comparable with the "
            "other countries. Converted at ECB 09.09.2026, EUR 1 = HUF 363.95; VAT "
            "27%."
        ),
    ),
    (
        "IT-RFI-EXTRA-PMdA-2026",
        "it_rfi_extra_pmda_2026",
        "Used",
        "",
        (
            "Listino tariffario servizi extra-PMdA 2025-2029 (2026 values) with "
            "Perimetro stazioni passeggeri servizi extra-PMdA"
        ),
        "RFI",
        2026,
        2026,
        "EUR",
        "network_statement",
        (
            " "
            "https://www.rfi.it/content/dam/rfi/offerta/sistema-tariffario-2025-2029/extra_pmda/Listino%20Tariffario%20Servizi%20Extra%20PMdA.pdf"
        ),
        "2026-09-10",
        (
            "Per stop, the sum of the station components a commercial long-distance "
            "or night train (OA LP) pays: reception areas 1.44, toilets 0.08, "
            "information (IaP) base/standard/top 0.31/1.08/4.19, PRM assistance "
            "8.08 where the station is in PRM circle F1 or F2. Which components "
            "apply per station comes from the perimeter list. 22% VAT is an "
            "assumption — neither RFI document states a rate."
        ),
    ),
    (
        "NL-ProRail-NS-2026-Transfer",
        "nl_prorail_ns_2026",
        "Used",
        "",
        (
            "Netverklaring 2026 (definitieve versie 13.12.2024), Bijlage 25 and the "
            "transfer tariff table (p. 120)"
        ),
        "ProRail",
        2024,
        2026,
        "EUR",
        "network_statement",
        (
            " "
            "https://www.prorail.nl/siteassets/homepage/samenwerken/vervoerders/documenten/2026-netverklaring/netverklaring-2026---definitieve-versie-d.d.-13-december-2024.pdf"
        ),
        "2026-09-10",
        (
            "Station transfer facility tariff by station class (halte, basis, plus, "
            "mega, kathedraal) and train-stop code A/B/C. Code C assumed for "
            "long-distance and night trains; the actual code is assigned per train "
            "number series. 21% VAT."
        ),
    ),
    (
        "PL-PKP-SA-OIU-2025-2026-ANNEX1-ANNEX4",
        "pl_pkp_sa_oiu_2026",
        "Used",
        "",
        (
            "Regulamin dostępu do obiektów infrastruktury usługowej 2025/2026 — "
            "Załącznik 1 (kategorie) and Załącznik 4 (cennik)"
        ),
        "PKP S.A.",
        2025,
        2026,
        "PLN",
        "network_statement",
        (
            " "
            "https://www.pkp.pl/images/download/stacje/2026/Zacznik%20nr%204%20do%20Regulaminu%20RRJ%202025_2026.pdf"
        ),
        "2026-09-10",
        (
            "Passenger-station access per realised stop, long-distance traffic, by "
            "station category; Premium stations carry an individual tariff. "
            "Converted at ECB 09.09.2026, EUR 1 = PLN 4.3153; VAT 23%."
        ),
    ),
    (
        "PL-PKP-PLK-OIU-2025-2026-ANNEX7",
        "pl_pkp_plk_oiu_2026",
        "Used",
        "",
        "Regulamin OIU 2025/2026 PKP PLK, Załącznik 7",
        "PKP Polskie Linie Kolejowe S.A.",
        2025,
        2026,
        "PLN",
        "network_statement",
        "https://www.plk-sa.pl/",
        "2026-09-10",
        (
            "Stations managed by PKP PLK rather than PKP S.A.: 10.49 PLN per "
            "ordered and realised stop including origin and destination. Used for "
            "Włoszczowa Północ. Converted at ECB 09.09.2026, EUR 1 = PLN 4.3153; "
            "VAT 23%."
        ),
    ),
    (
        "PT-IP-DR-2026-ESTACOES",
        "pt_ip_dr_2026",
        "Used",
        "",
        "Diretório da Rede 2026, 2.a Adenda, Anexo 7.3.2 A",
        "Infraestruturas de Portugal",
        2025,
        2026,
        "EUR",
        "network_statement",
        (
            " "
            "https://servicos.infraestruturasdeportugal.pt/sites/default/files/2%C2%AA%20Adenda%20Diretorio%20da%20Rede%202026_signed.pdf"
        ),
        "2026-09-10",
        (
            "Station use tariff per commercial stop by station category A-D; "
            "additional services (operational facilities, consumption, commercial "
            "information) excluded. VAT 23%."
        ),
    ),
    (
        "RO-CFR-DRR-2026-ANNEX-26A",
        "ro_cfr_drr_2026",
        "Used",
        "",
        "Network Statement 2026, Annex 26.a, service 2.1",
        "CFR S.A.",
        2025,
        2026,
        "RON",
        "network_statement",
        "https://cfr.ro/files/ddr/EN%202026/NS%202026.pdf",
        "2026-09-10",
        (
            "2.45 RON per commercial passenger stop, valid from 01.04.2025, "
            "covering electricity, passenger-information boards and public address. "
            "Converted at ECB 09.09.2026, EUR 1 = RON 5.2546; VAT 21%."
        ),
    ),
    (
        "SE-TRAFIKVERKET-NS-2026-7.3.2",
        "se_trv_ns_2026",
        "Used",
        "",
        "Network Statement 2026, section 7.3.2",
        "Trafikverket",
        2024,
        2026,
        "SEK",
        "network_statement",
        (
            " "
            "https://www.trafikverket.se/en/operations/Operations-railway/Network-Statement/"
        ),
        "2026-09-10",
        (
            "Track adjacent to a platform is part of the train path and platform "
            "access for passenger exchange carries no separate charge. Every "
            "Swedish stop is recorded as not levied."
        ),
    ),
    (
        "SK-ZSR-NS-2026-MEASURE-2-2018-ANNEX2",
        "sk_zsr_ns_2026",
        "Used",
        "",
        (
            "Network Statement 2026 — Annex 2.3.3.A (station categories) and Annex "
            "5.2.B (prices), per Measure 2/2018"
        ),
        "ŽSR",
        2025,
        2026,
        "EUR",
        "network_statement",
        (
            " "
            "https://www.zsr.sk/en/railway-undertaking/infrastructure/network-statement/2026"
        ),
        "2026-09-10",
        (
            "Usz1 passenger-station access per scheduled stop by station category "
            "A-C, for 'other passenger trains' (an Os stopping train pays a lower "
            "rate); origin and destination count as stops. VAT 23%."
        ),
    ),
    (
        "TR-TCDD-NS-2026-ANNEX-6.3.2",
        "tr_tcdd_ns_2026",
        "Used",
        "",
        (
            "Network Statement 2026 — Annex 6.3.2 (fees) and Annex 3.3.1.3 (station "
            "characteristics)"
        ),
        "TCDD",
        2025,
        2026,
        "TRY",
        "network_statement",
        (
            " "
            "https://static.tcdd.gov.tr/webfiles/userfiles/files/sebekebildirimi/2026/ing/63210.pdf"
        ),
        "2026-09-10",
        (
            "Passenger-station usage fee for a main-line train at an intermediate "
            "stop (max. 10 min), station class A or B derived from TCDD's scoring "
            "of passenger volume, status, waiting rooms, security and platforms; "
            "origin/destination stops and HST trains have other fees. Converted at "
            "08.09.2026, EUR 1 = TRY 56.2814; VAT 20%."
        ),
    ),
    (
        "UA-UZ-PASSENGER-MARKET-2026",
        "ua_uz_2026",
        "Used",
        "",
        (
            "Ukrzaliznytsia passenger-market framework 2026 (research conclusion, "
            "no tariff document)"
        ),
        "JSC Ukrzaliznytsia",
        2026,
        2026,
        "UAH",
        "manual_transcription",
        "",
        "2026-09-10",
        (
            "No published operator-facing per-stop station access tariff was "
            "identified; under the 2026 state order UZ is the sole domestic carrier "
            "and prices station services to passengers, not to operators. Every "
            "Ukrainian stop is recorded as not levied. This is a research "
            "conclusion, not a document — revisit if the market opens."
        ),
    ),
    # Further sources go here. One tuple per document, same column order
    # as REGISTER_COLUMNS. Keep the id stable once a value in 02 cites it.
]

## The values carried over from the retired curated catalog

Thirteen stations carried a `stop_charge_eur` in `db/dev/seed.py`'s curated
list, which the stop classification pipeline replaced. `seed.py` attributed
them to "Illustrative / internal estimate" — they are **not** published
tariffs, and every one should be replaced by a sourced figure. Registered here
so the values are traceable rather than silently inherited.

In [3]:
register_rows += [
    (
        "ILLUSTRATIVE-CURATED",
        "illustrative_curated",
        "Used",
        "",
        "Curated stop catalog placeholder charges (retired 2026-08-18)",
        "Back-on-Track (internal)",
        2026,
        2026,
        "EUR",
        "manual_transcription",
        "db/dev/seed.py, _STOP_INFRASTRUCTURES_CANONICAL before its removal",
        _R,
        (
            "NOT a published tariff. Illustrative internal estimates, kept only so "
            "the figures are not lost. Replace each one with a sourced value."
        ),
    ),
]

## Write and validate

In [4]:
def write_data(name: str, columns: list[str], rows: list[dict]) -> None:
    """STDLIB-ONLY writer, shared by both charge notebooks."""
    path = DATA_DIR / name
    with open(path, "w", encoding="utf-8", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    print(f"  {name}: {len(rows)} rows")


rows = [dict(zip(REGISTER_COLUMNS, row, strict=True)) for row in register_rows]

ids = [row["source_id"] for row in rows]
duplicates = sorted({i for i in ids if ids.count(i) > 1})
if duplicates:
    raise ValueError(f"duplicate source_id: {duplicates}")

missing_files = [
    row["url_or_file"]
    for row in rows
    if row["downloaded"] == "x" and not (SOURCES_DIR / row["url_or_file"]).is_file()
]
if missing_files:
    print(f"\n  WARNING: marked downloaded but absent from sources/: {missing_files}")

write_data("sources_register.csv", REGISTER_COLUMNS, rows)
print(
    f"register: {len(rows)} sources, "
    f"{sum(1 for r in rows if r['used'] == 'Used')} in use, "
    f"{sum(1 for r in rows if r['downloaded'] == 'x')} documents on disk"
)

  sources_register.csv: 21 rows
register: 21 sources, 20 in use, 0 documents on disk
